In [1]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [2]:
from sklearn.preprocessing import MinMaxScaler

In [3]:
from sklearn.metrics import root_mean_squared_error

In [4]:
ross_df = pd.read_csv('train.csv')
store_df = pd.read_csv('store.csv')
test_df = pd.read_csv('test.csv')
submission_df = pd.read_csv('sample_submission.csv')

C:\Users\aadar\AppData\Local\Temp\ipykernel_1184\911464789.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  ross_df = pd.read_csv('train.csv')


In [ ]:
ross_df

In [ ]:
store_df

In [5]:
merged_df = ross_df.merge(store_df,how='left',on='Store')
merged_test_df = test_df.merge(store_df,how='left',on='Store')

In [6]:
def split_date(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['day'] = df.Date.dt.day
    df['month'] = df.Date.dt.month
    df['year'] = df.Date.dt.year
    df['weekofyear'] = df.Date.dt.isocalendar().week

In [7]:
split_date(merged_df)
split_date(merged_test_df)

In [ ]:
merged_df

In [8]:
merged_df = merged_df[merged_df['Open']==1].copy()

In [9]:
def comp_months(df):
    df['competitionOpen'] = 12*(df.year-df.CompetitionOpenSinceYear) + (df.month-df.CompetitionOpenSinceMonth)
    df['competitionOpen'] = df['competitionOpen'].map(lambda x:0 if x<0 else x).fillna(0)

In [10]:
comp_months(merged_df)
comp_months(merged_test_df)

In [11]:
def check_promo_month(row):
    month2str = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',              
                 7:'Jul', 8:'Aug', 9:'Sept', 10:'Oct', 11:'Nov', 12:'Dec'}
    try:
        months = (row['PromoInterval'] or '').split(',')
        if row['Promo2Open'] and month2str[row['month']] in months:
            return 1
        else:
            return 0
    except Exception:
        return 0

def promo_cols(df):
    # Months since Promo2 was open
    df['Promo2Open'] = 12 * (df.year - df.Promo2SinceYear) +  (df.weekofyear - df.Promo2SinceWeek)*7/30.5
    df['Promo2Open'] = df['Promo2Open'].map(lambda x: 0 if x < 0 else x).fillna(0) * df['Promo2']
    # Whether a new round of promotions was started in the current month
    df['IsPromo2Month'] = df.apply(check_promo_month, axis=1) * df['Promo2']

In [12]:
promo_cols(merged_df)
promo_cols(merged_test_df)

In [ ]:
merged_df

In [13]:
def get_store_sales_statistics(df):
    mean = df.groupby('Store')['Sales'].mean()
    std = df.groupby('Store')['Sales'].std() # Fixed this to .std() for you!
    
    # Step 1: Create the dataframe (no dot at the end, so no crash)
    mean_dataframe = pd.DataFrame(mean)
    std_dataframe = pd.DataFrame(std)
    
    # Step 2: Reset the index on a new line
    mean_dataframe = mean_dataframe.reset_index().rename(columns={"Sales": "SalesMean"})
    std_dataframe = std_dataframe.reset_index().rename(columns={"Sales": "SalesStd"})

    df = pd.merge(df,mean_dataframe, on='Store', how='left')
    df = pd.merge(df,std_dataframe, on='Store', how='left')

    return df

In [14]:
def get_sales_level_groups(df):
    q1 = df.SalesMean.quantile(0.25)
    q2 = df.SalesMean.quantile(0.5)
    q3 = df.SalesMean.quantile(0.75)

    df['storegroup1'] = (df.SalesMean < q1).astype(int)
    df['storegroup2'] = ((df.SalesMean>=q1)&(df.SalesMean<q2)).astype(int)
    df['storegroup3'] = ((df.SalesMean>=q2)&(df.SalesMean<q3)).astype(int)
    df['storegroup4'] = (df.SalesMean > q3).astype(int)
    df['StoreGroup'] = df['storegroup1'] + df['storegroup2']*2+ df['storegroup3']*3+df['storegroup4']*4
    df.drop(['storegroup1','storegroup2','storegroup3','storegroup4'],axis=1, inplace=True)
    return df

In [15]:
merged_df = get_store_sales_statistics(merged_df)
merged_df = get_sales_level_groups(merged_df)

In [16]:
columns_to_transfer = ['Store', 'SalesMean', 'SalesStd', 'StoreGroup']
store_lookup_table = merged_df[columns_to_transfer].drop_duplicates(subset=['Store'])
merged_test_df = pd.merge(merged_test_df, store_lookup_table, on='Store', how='left')

In [ ]:
merged_test_df.columns

In [ ]:
merged_df

In [17]:
merged_df.corr(numeric_only=True)['Sales'].sort_values(ascending=False)

Sales                        1.000000
Customers                    0.823597
SalesMean                    0.775808
StoreGroup                   0.645522
SalesStd                     0.600975
Promo                        0.368145
Promo2SinceWeek              0.095311
weekofyear                   0.074472
month                        0.073600
SchoolHoliday                0.038617
year                         0.036169
CompetitionOpenSinceYear     0.016101
Store                        0.007710
competitionOpen             -0.003196
Promo2SinceYear             -0.034713
CompetitionDistance         -0.036396
CompetitionOpenSinceMonth   -0.043489
day                         -0.051849
Promo2Open                  -0.060761
IsPromo2Month               -0.065369
Promo2                      -0.127596
DayOfWeek                   -0.178736
Open                              NaN
Name: Sales, dtype: float64

In [18]:
merged_df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval', 'day', 'month', 'year',
       'weekofyear', 'competitionOpen', 'Promo2Open', 'IsPromo2Month',
       'SalesMean', 'SalesStd', 'StoreGroup'],
      dtype='object')

In [ ]:
input_cols = ['Store', 'DayOfWeek','Date', 'Promo', 'StateHoliday', 'SchoolHoliday', 
              'StoreType', 'Assortment', 'CompetitionDistance', 'competitionOpen', 
              'day', 'month', 'year', 'weekofyear',  'Promo2', 
              'Promo2Open', 'IsPromo2Month','SalesMean','SalesStd','StoreGroup']
target_col = 'Sales'

In [ ]:
inputs = merged_df[input_cols].copy()
targets = merged_df[target_col].copy()

In [ ]:
test_inputs = merged_test_df[input_cols].copy()

In [19]:
numeric_cols = ['Store','DayOfWeek', 'Promo', 'SchoolHoliday', 
              'CompetitionDistance', 'competitionOpen', 'Promo2', 'Promo2Open', 'IsPromo2Month',
              'day', 'month', 'year', 'weekofyear','SalesMean','SalesStd','StoreGroup' ]
categorical_cols = ['StateHoliday', 'StoreType', 'Assortment']

In [ ]:
merged_df[numeric_cols].isna().sum()

In [20]:
merged_test_df[numeric_cols].isna().sum()

Store                   0
DayOfWeek               0
Promo                   0
SchoolHoliday           0
CompetitionDistance    96
competitionOpen         0
Promo2                  0
Promo2Open              0
IsPromo2Month           0
day                     0
month                   0
year                    0
weekofyear              0
SalesMean               0
SalesStd                0
StoreGroup              0
dtype: int64

In [21]:
max_distance = merged_df.CompetitionDistance.max()
max_distance

np.float64(75860.0)

In [22]:
merged_df['CompetitionDistance'].fillna(max_distance*2, inplace=True)
merged_test_df['CompetitionDistance'].fillna(max_distance*2, inplace=True)

C:\Users\aadar\AppData\Local\Temp\ipykernel_1184\501010057.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['CompetitionDistance'].fillna(max_distance*2, inplace=True)
C:\Users\aadar\AppData\Local\Temp\ipykernel_1184\501010057.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as

In [23]:
scaler = MinMaxScaler().fit(merged_df[numeric_cols])
merged_df[numeric_cols] = scaler.transform(merged_df[numeric_cols])
merged_test_df[numeric_cols] = scaler.transform(merged_test_df[numeric_cols])

In [24]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
inputs[categorical_cols].info()

In [25]:
# Force all categorical columns to be strings
merged_df[categorical_cols] = merged_df[categorical_cols].astype(str)

In [27]:
encoder = OneHotEncoder(sparse_output=False,handle_unknown='ignore').fit(merged_df[categorical_cols])
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

In [ ]:
encoded_cols

In [28]:
merged_df[encoded_cols] = encoder.transform(merged_df[categorical_cols])
merged_test_df[encoded_cols] = encoder.transform(merged_test_df[categorical_cols])

In [30]:
merged_df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval', 'day', 'month', 'year',
       'weekofyear', 'competitionOpen', 'Promo2Open', 'IsPromo2Month',
       'SalesMean', 'SalesStd', 'StoreGroup', 'StateHoliday_0',
       'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c', 'StoreType_a',
       'StoreType_b', 'StoreType_c', 'StoreType_d', 'Assortment_a',
       'Assortment_b', 'Assortment_c'],
      dtype='object')

In [29]:
train_size = int(.75* len(merged_df))
train_size

633294

In [31]:
sorted_df = merged_df.sort_values('Date')
train_df,val_df = sorted_df[:train_size],sorted_df[train_size:]
len(val_df)

211098

In [253]:
merged_test_df = merged_test_df[numeric_cols+encoded_cols]

In [32]:
x_train = train_df[numeric_cols+encoded_cols]
x_val = val_df[numeric_cols+encoded_cols]
y_train = train_df['Sales']
y_val = val_df['Sales']

In [254]:
merged_test_df.to_parquet('new_test_inputs.parquet')

In [ ]:
x_train.to_parquet('new_x_train.parquet')
x_val.to_parquet('new_x_val.parquet')
y_train.to_frame().to_parquet('new_y_train.parquet')
y_val.to_frame().to_parquet('new_y_val.parquet')

#### #Load saved DF

In [255]:
x_test = pd.read_parquet('new_test_inputs.parquet')

In [ ]:
x_train = pd.read_parquet('new_x_train.parquet')
x_val = pd.read_parquet('new_x_val.parquet')
y_train = pd.read_parquet('new_y_train.parquet')
y_val = pd.read_parquet('new_y_val.parquet')


In [37]:
from xgboost import XGBRegressor

In [84]:
model = XGBRegressor(random_state=42,n_jobs=-1,n_estimators=100,learning_rate=0.4,max_depth=5)

In [85]:
model.fit(x_train,y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [86]:
pred = model.predict(x_val)
root_mean_squared_error(y_val,pred)

1137.2462158203125

In [124]:
model_2 = XGBRegressor(random_state = 42, n_jobs=-1, n_estimators=400,learning_rate=0.1,max_depth=5)

In [125]:
model_2.fit(x_train,y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [126]:
pred2 = model_2.predict(x_val)
root_mean_squared_error(y_val,pred2)

1122.982177734375

In [247]:
model_3 = XGBRegressor(random_state=42,n_jobs=-1,n_estimators=1000,learning_rate=0.1,max_depth=6,subsample=0.95,colsample_bytree=0.7)

In [248]:
model_3.fit(x_train,y_train)
pred3 = model_3.predict(x_val)
root_mean_squared_error(pred3,y_val)

1088.2039794921875

In [265]:
model.save_model('xgboost_rossmann_sales_model.json')

In [250]:
importance_df = pd.DataFrame({
    'feature': x_train.columns,
    'importance': model_3.feature_importances_
}).sort_values('importance', ascending=False)
importance_df

,feature,importance
15,StoreGroup,0.551814
13,SalesMean,0.134985
2,Promo,0.128113
14,SalesStd,0.023954
1,DayOfWeek,0.020829
21,StoreType_b,0.014319
20,StoreType_a,0.013501
9,day,0.011891
12,weekofyear,0.011482
25,Assortment_b,0.010175


In [256]:
x_test.columns

Index(['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'CompetitionDistance',
       'competitionOpen', 'Promo2', 'Promo2Open', 'IsPromo2Month', 'day',
       'month', 'year', 'weekofyear', 'SalesMean', 'SalesStd', 'StoreGroup',
       'StateHoliday_0', 'StateHoliday_a', 'StateHoliday_b', 'StateHoliday_c',
       'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d',
       'Assortment_a', 'Assortment_b', 'Assortment_c'],
      dtype='object')

In [257]:
test_pred = model_3.predict(x_test)

In [258]:
submission_df['Sales'] = test_pred

In [262]:
test_df.Open.isna().sum()

np.int64(11)

In [263]:
submission_df['Sales'] = submission_df['Sales'] * test_df.Open.fillna(1.)

In [264]:
submission_df.to_csv('submission.csv', index=None)